In [2]:



import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
import joblib
import os


INPUT_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Dataset\\engineered_dataset.csv"
OUTPUT_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\final"
ENCODERS_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\encoders"
SCALERS_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\scalers"

# Create directories if they don't exist
for path in [OUTPUT_PATH, ENCODERS_PATH, SCALERS_PATH]:
    os.makedirs(path, exist_ok=True)

print("Configuration set ✅")


Configuration set ✅


In [3]:

try:
    df = pd.read_csv(INPUT_PATH)
    print(f"✓ Loaded dataset: {df.shape}")
except FileNotFoundError:
    raise FileNotFoundError(f"❌ File not found at {INPUT_PATH}")

# Create a working copy
df_clean = df.copy()


✓ Loaded dataset: (606000, 27)


In [4]:


missing_before = df_clean.isnull().sum().sum()
print(f"Missing values before cleaning: {missing_before}")

# Separate numeric and categorical columns
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric columns: {len(numeric_cols)}, Categorical columns: {len(categorical_cols)}")

if missing_before > 0:
    # Impute numeric columns with median
    num_imputer = SimpleImputer(strategy='median')
    df_clean[numeric_cols] = num_imputer.fit_transform(df_clean[numeric_cols])
    
    # Impute categorical columns with most frequent
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df_clean[categorical_cols] = cat_imputer.fit_transform(
        df_clean[categorical_cols].values.reshape(-1, len(categorical_cols))
    )
    
    # Save imputers
    joblib.dump(num_imputer, f"{SCALERS_PATH}numeric_imputer.pkl")
    joblib.dump(cat_imputer, f"{ENCODERS_PATH}categorical_imputer.pkl")
    print("✓ Imputers saved")

missing_after = df_clean.isnull().sum().sum()
print(f"Missing values after cleaning: {missing_after}")


Missing values before cleaning: 0
Numeric columns: 18, Categorical columns: 9
Missing values after cleaning: 0


In [5]:


duplicates_before = df_clean.duplicated().sum()
print(f"Duplicates before: {duplicates_before}")

if duplicates_before > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"✓ Removed {duplicates_before} duplicate rows")

print(f"Dataset shape after deduplication: {df_clean.shape}")


Duplicates before: 0
Dataset shape after deduplication: (606000, 27)


In [10]:
categorical_to_encode = [
    'product_category', 'shipping_type', 'material_type',
    'packaging_type', 'recyclability_category', 'supplier_region'
]

categorical_to_encode = [col for col in categorical_to_encode if col in df_clean.columns]

label_encoders = {}

for col in categorical_to_encode:
    le = LabelEncoder()
    df_clean[f'{col}_encoded'] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le
    
    # Save encoder
    joblib.dump(le, f"C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\encoders\\{col}_encoder.pkl")
    
    print(f"Encoded {col} ({len(le.classes_)} unique values)")

print(f"✓ Saved {len(label_encoders)} encoders")


Encoded product_category (7 unique values)
Encoded shipping_type (3 unique values)
Encoded material_type (4 unique values)
Encoded packaging_type (6 unique values)
Encoded recyclability_category (2 unique values)
Encoded supplier_region (6 unique values)
✓ Saved 6 encoders


In [ ]:

features_to_normalize = [
    'product_weight_kg', 'fragility_index', 'recyclability_percent', 'biodegradation_days',
    'carbon_footprint', 'co2_emission_per_kg', 'load_handling_score', 'moisture_resistance',
    'thermal_resistance', 'cost_per_unit_usd', 'reusability_percent', 'recycled_content_percent',
    'waste_reduction_impact', 'co2_impact_index', 'cost_efficiency_index',
    'material_suitability_score', 'overall_sustainability_score', 'compatibility_score'
]

# Filter existing columns
features_to_normalize = [col for col in features_to_normalize if col in df_clean.columns]

# Check for missing engineered columns
for col in ['overall_sustainability_score']:
    if col not in df_clean.columns:
        print(f"⚠ '{col}' missing. Calculating now...")
        co2_score_inverted = 100 - df_clean['co2_impact_index']
        df_clean[col] = (
            0.40 * co2_score_inverted +
            0.30 * df_clean['cost_efficiency_index'] +
            0.30 * df_clean['material_suitability_score']
        )
        print(f"✓ '{col}' created")

# Normalize using MinMaxScaler
scaler = MinMaxScaler()
df_clean[features_to_normalize] = scaler.fit_transform(df_clean[features_to_normalize])

# Save scaler
joblib.dump(scaler, f"C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\scalers\\feature_scaler.pkl")
print("✓ Features normalized and scaler saved")

# Sample preview
df_clean[features_to_normalize].head(3)


⚠ 'overall_sustainability_score' missing. Calculating now...
✓ 'overall_sustainability_score' created
✓ Features normalized and scaler saved


,product_weight_kg,fragility_index,recyclability_percent,biodegradation_days,carbon_footprint,co2_emission_per_kg,load_handling_score,moisture_resistance,thermal_resistance,cost_per_unit_usd,reusability_percent,recycled_content_percent,waste_reduction_impact,co2_impact_index,cost_efficiency_index,material_suitability_score,compatibility_score
0,0.030749,0.25,0.955556,0.000871,0.139881,0.087542,0.555556,0.444444,0.250,0.078866,0.474227,0.794521,0.569231,0.081654,0.994506,0.789474,1.000000
1,0.030749,0.25,1.000000,0.000189,0.062500,0.013468,0.222222,0.222222,0.125,0.062549,0.288660,0.986301,0.676923,0.021102,0.909820,0.157895,0.235294
2,0.030749,0.25,0.711111,1.000000,1.000000,1.000000,0.777778,0.888889,0.875,0.963092,1.000000,0.780822,0.938462,1.000000,0.067731,0.578947,0.529412


In [12]:


ml_features = features_to_normalize + [f'{col}_encoded' for col in categorical_to_encode]

# Reload original dataset for raw targets
df_original = pd.read_csv(INPUT_PATH)

# Features
X = df_clean[ml_features].copy()

# Targets
y_cost = df_original['cost_per_unit_usd'].copy()
y_co2 = df_original['carbon_footprint'].copy()
y_sustainability = df_clean['overall_sustainability_score'].copy()  # normalized

print(f"Feature matrix shape: {X.shape}")
print(f"Target shapes: cost={y_cost.shape}, co2={y_co2.shape}, sustainability={y_sustainability.shape}")


Feature matrix shape: (606000, 23)
Target shapes: cost=(606000,), co2=(606000,), sustainability=(606000,)


In [14]:


# Save full cleaned dataset
df_clean.to_csv(f"C:\\Users\\sneha\\Desktop\\ecopackai\\Dataset\\cleaned_dataset.csv", index=False)

# Save ML-ready datasets
X.to_csv(f"{OUTPUT_PATH}X_features.csv", index=False)
y_cost.to_csv(f"C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_ready\\y_cost.csv", index=False, header=['cost_per_unit_usd'])
y_co2.to_csv(f"C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_ready\\y_co2.csv", index=False, header=['carbon_footprint'])
y_sustainability.to_csv(f"C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_ready\\y_sustainability.csv", index=False, header=['overall_sustainability_score'])

# Save feature names
import json
feature_names = {
    'numerical_features': features_to_normalize,
    'categorical_features': categorical_to_encode,
    'encoded_features': [f'{col}_encoded' for col in categorical_to_encode],
    'all_ml_features': ml_features
}

with open(f"C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\processed\\feature_names.json", 'w') as f:
    json.dump(feature_names, f, indent=2)

print("✓ Processed data and metadata saved")


✓ Processed data and metadata saved


In [16]:
# ============================================
# 9. DATA QUALITY REPORT
# ============================================

report = {
    'original_shape': df.shape,
    'cleaned_shape': df_clean.shape,
    'rows_removed': df.shape[0] - df_clean.shape[0],
    'missing_values_handled': missing_before,
    'duplicates_removed': duplicates_before,
    'categorical_features_encoded': len(categorical_to_encode),
    'numerical_features_normalized': len(features_to_normalize),
    'total_ml_features': len(ml_features),
    'target_variables': ['cost_prediction', 'co2_prediction', 'sustainability_score']
}

# Print summary
for key, value in report.items():
    print(f"{key}: {value}")

# Save report
report_path = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\docs\\data_cleaning_report.txt"
os.makedirs("docs", exist_ok=True)
with open(report_path, 'w') as f:
    f.write("EcoPackAI - Data Cleaning Report\n")
    f.write("="*60 + "\n\n")
    for key, value in report.items():
        f.write(f"{key}: {value}\n")

print(f"\n✓ Data cleaning report saved at {report_path}")


original_shape: (606000, 27)
cleaned_shape: (606000, 34)
rows_removed: 0
missing_values_handled: 0
duplicates_removed: 0
categorical_features_encoded: 6
numerical_features_normalized: 17
total_ml_features: 23
target_variables: ['cost_prediction', 'co2_prediction', 'sustainability_score']

✓ Data cleaning report saved at C:\Users\sneha\Desktop\ecopackai\Data\docs\data_cleaning_report.txt
